# HW 1: Classic ML Text Classification

News topic classification on lenta-ru-news with bag-of-words and TF-IDF.

## Setup

1. Install [uv](https://docs.astral.sh/uv/getting-started/installation/)
2. `uv sync` from repo root
3. `uv run jupyter notebook`
4. Dataset downloads automatically on first run


In [ ]:
import os

os.chdir(os.path.join(os.path.dirname(os.getcwd()), "") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd())

# Always work from repo root
REPO_ROOT = os.getcwd()
print(f"Working directory: {REPO_ROOT}")
assert os.path.exists("pyproject.toml"), "Run this notebook from the repo root or the notebooks/ folder"

### Imports


In [ ]:
import re
import random
import string
import warnings

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from joblib import dump

import razdel
import pymorphy3
from corus import load_lenta
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

print("Imports OK")

## 1. Data Loading


In [ ]:
RAW_DATA_PATH = os.path.join(REPO_ROOT, "data", "raw", "hw_1", "lenta-ru-news.csv.gz")

if not os.path.exists(RAW_DATA_PATH):
    import urllib.request
    url = "https://github.com/yutkin/Lenta.Ru-News-Dataset/releases/download/v1.0/lenta-ru-news.csv.gz"
    print(f"Downloading dataset to {RAW_DATA_PATH}...")
    os.makedirs(os.path.dirname(RAW_DATA_PATH), exist_ok=True)
    urllib.request.urlretrieve(url, RAW_DATA_PATH)
    print("Done.")
else:
    print(f"Dataset already exists at {RAW_DATA_PATH}")

In [ ]:
records = []
for record in load_lenta(RAW_DATA_PATH):
    records.append({
        "title": record.title,
        "text": record.text,
        "topic": record.topic,
    })

df_full = pd.DataFrame(records)
print(f"Total records loaded: {len(df_full):,}")
df_full.head()

## 2. Data Preparation

### Sampling

Take a stratified 100k sample to keep class proportions.


In [ ]:
SAMPLE_SIZE = 100_000

# Drop nulls and empties
df_full = df_full.dropna(subset=["title", "text", "topic"])
df_full = df_full[df_full["text"].str.strip().astype(bool)].reset_index(drop=True)

print(f"Records after cleaning nulls/empties: {len(df_full):,}")
print(f"Number of unique topics: {df_full['topic'].nunique()}")
print()
print(df_full["topic"].value_counts())

In [ ]:
# Stratified 100k sample
df = df_full.groupby("topic", group_keys=False).apply(
    lambda x: x.sample(
        n=min(len(x), int(SAMPLE_SIZE * len(x) / len(df_full))),
        random_state=RANDOM_STATE,
    )
).reset_index(drop=True)

if len(df) > SAMPLE_SIZE:
    df = df.sample(n=SAMPLE_SIZE, random_state=RANDOM_STATE).reset_index(drop=True)

print(f"Sampled dataset size: {len(df):,}")
print()
print(df["topic"].value_counts())

### EDA


In [ ]:
df["text_len"] = df["text"].str.len()
df["word_count"] = df["text"].str.split().str.len()

print(df[["text_len", "word_count"]].describe())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Topic distribution
df["topic"].value_counts().plot.barh(ax=axes[0])
axes[0].set_title("Topic Distribution")
axes[0].set_xlabel("Count")

# Text length distribution
df["text_len"].clip(upper=df["text_len"].quantile(0.99)).hist(bins=50, ax=axes[1])
axes[1].set_title("Text Length (chars, clipped at 99th percentile)")
axes[1].set_xlabel("Characters")

# Word count distribution
df["word_count"].clip(upper=df["word_count"].quantile(0.99)).hist(bins=50, ax=axes[2])
axes[2].set_title("Word Count (clipped at 99th percentile)")
axes[2].set_xlabel("Words")

plt.tight_layout()
plt.show()

### Data quality check

Following lecture best practices: check duplicates, conflicting labels, bad texts, and outliers.


In [ ]:
# Duplicates
n_dup = df.duplicated("text").sum()
print(f"Duplicate texts: {n_dup}")

# Conflicting labels: same text, different topic
n_conflict = (df.groupby("text")["topic"].nunique() > 1).sum()
print(f"Texts with conflicting topic labels: {n_conflict}")

# Drop duplicates, keeping first occurrence
df = df.drop_duplicates("text").reset_index(drop=True)
print(f"After dedup: {len(df):,}")


In [ ]:
# Bad texts: empty, only punctuation/symbols, only digits
def is_bad_text(s):
    s0 = s.strip()
    if len(s0) == 0:
        return True
    if re.fullmatch(r"[^a-zA-Za-z\u0400-\u04FF]+", s0):
        return True
    if re.fullmatch(r"\d+", s0):
        return True
    return False

bad_mask = df["text"].apply(is_bad_text)
print(f"Bad texts: {bad_mask.sum()} ({bad_mask.mean():.2%})")
df = df[~bad_mask].reset_index(drop=True)


In [ ]:
# Length outliers (clip at 99.5th percentile)
upper = df["word_count"].quantile(0.995)
too_long = df["word_count"] > upper
print(f"Texts longer than {upper:.0f} words: {too_long.sum()}")
df = df[~too_long].reset_index(drop=True)
print(f"Clean dataset: {len(df):,}")


### Text Preprocessing

Pipeline: lowercase -> remove URLs/emails/special chars -> tokenize (razdel) -> lemmatize (pymorphy3) -> filter short tokens.

Using lemmatization over stemming because Russian morphology is complex and pymorphy3 gives proper dictionary forms.


In [ ]:
morph = pymorphy3.MorphAnalyzer()

# Compile regex patterns once for speed
RE_URL = re.compile(r"https?://\S+|www\.\S+")
RE_EMAIL = re.compile(r"\S+@\S+\.\S+")
RE_NONALPHA = re.compile(r"[^a-zA-Z\u0400-\u04FF\s]")
RE_MULTISPACE = re.compile(r"\s+")


def preprocess_text(text):
    """Full preprocessing pipeline for a single text."""
    # Lowercase
    text = text.lower()
    # Remove URLs and emails
    text = RE_URL.sub(" ", text)
    text = RE_EMAIL.sub(" ", text)
    # Remove non-alphabetic characters (keep Cyrillic and Latin)
    text = RE_NONALPHA.sub(" ", text)
    # Normalize whitespace
    text = RE_MULTISPACE.sub(" ", text).strip()
    # Tokenize with razdel
    tokens = [token.text for token in razdel.tokenize(text)]
    # Lemmatize and filter
    lemmas = []
    for token in tokens:
        if len(token) < 2:
            continue
        lemma = morph.parse(token)[0].normal_form
        lemmas.append(lemma)
    return " ".join(lemmas)


# Test on a sample
sample_text = df["text"].iloc[0]
print("Original:", sample_text[:200])
print()
print("Processed:", preprocess_text(sample_text)[:200])

In [ ]:
# Combine title + text for better signal
tqdm.pandas(desc="Preprocessing texts")
df["full_text"] = df["title"].fillna("") + " " + df["text"].fillna("")
df["processed_text"] = df["full_text"].progress_apply(preprocess_text)

In [ ]:
empty_mask = df["processed_text"].str.strip() == ""
print(f"Empty texts after preprocessing: {empty_mask.sum()}")
df = df[~empty_mask].reset_index(drop=True)
print(f"Final dataset size: {len(df):,}")

In [ ]:
# Save interim data
interim_path = os.path.join(REPO_ROOT, "data", "interim", "hw_1", "df_preprocessed.parquet")
df.to_parquet(interim_path, index=False)
print(f"Saved preprocessed data to {interim_path}")

### Train / Val / Test Split (60/20/20)


In [ ]:
X = df["processed_text"]
y = df["topic"]

# First split: 60% train, 40% temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.4, stratify=y, random_state=RANDOM_STATE
)

# Second split: 50/50 of temp = 20/20 of total
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=RANDOM_STATE
)

print(f"Train: {len(X_train):,} ({len(X_train)/len(X):.0%})")
print(f"Val:   {len(X_val):,} ({len(X_val)/len(X):.0%})")
print(f"Test:  {len(X_test):,} ({len(X_test)/len(X):.0%})")

In [ ]:
# Save splits (reused in HW2)
processed_dir = os.path.join(REPO_ROOT, "data", "processed", "hw_1")

for name, X_part, y_part in [
    ("train", X_train, y_train),
    ("val", X_val, y_val),
    ("test", X_test, y_test),
]:
    split_df = pd.DataFrame({"processed_text": X_part, "topic": y_part})
    path = os.path.join(processed_dir, f"{name}.parquet")
    split_df.to_parquet(path, index=False)
    print(f"Saved {name} split to {path}")

## 3. Dummy Baseline

Most-frequent class predictor as a sanity check.


In [ ]:
dummy = DummyClassifier(strategy="most_frequent", random_state=RANDOM_STATE)
dummy.fit(X_train, y_train)

y_pred_dummy = dummy.predict(X_val)
print(f"Dummy baseline accuracy: {accuracy_score(y_val, y_pred_dummy):.4f}")
print(f"Dummy baseline F1 (macro): {f1_score(y_val, y_pred_dummy, average='macro', zero_division=0):.4f}")
print()


## 4. LogisticRegression with Two Vectorizers

### CountVectorizer


In [ ]:
pipe_count = Pipeline([
    ("vectorizer", CountVectorizer()),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=RANDOM_STATE,
        solver="lbfgs",
        n_jobs=-1,
    )),
])

pipe_count.fit(X_train, y_train)

y_pred_count = pipe_count.predict(X_val)
print("=== CountVectorizer + LogisticRegression ===")
print(f"Accuracy: {accuracy_score(y_val, y_pred_count):.4f}")
print(f"F1 (macro): {f1_score(y_val, y_pred_count, average='macro'):.4f}")
print()
print(classification_report(y_val, y_pred_count))

### TfidfVectorizer

TF-IDF downweights common terms, usually helps for topic classification.


In [ ]:
pipe_tfidf = Pipeline([
    ("vectorizer", TfidfVectorizer()),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=RANDOM_STATE,
        solver="lbfgs",
        n_jobs=-1,
    )),
])

pipe_tfidf.fit(X_train, y_train)

y_pred_tfidf = pipe_tfidf.predict(X_val)
print("=== TfidfVectorizer + LogisticRegression ===")
print(f"Accuracy: {accuracy_score(y_val, y_pred_tfidf):.4f}")
print(f"F1 (macro): {f1_score(y_val, y_pred_tfidf, average='macro'):.4f}")
print()
print(classification_report(y_val, y_pred_tfidf))

## 5. Hyperparameter Tuning

RandomizedSearchCV with 3-fold stratified CV. Tuning vocab size, ngram range, frequency thresholds, and regularization.


In [ ]:
param_distributions = {
    "vectorizer__max_features": [10000, 30000, 50000, 100000, None],
    "vectorizer__ngram_range": [(1, 1), (1, 2)],
    "vectorizer__min_df": [1, 2, 5],
    "vectorizer__max_df": [0.8, 0.9, 1.0],
    "classifier__C": [0.01, 0.1, 1.0, 10.0],
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

In [ ]:
# Tune CountVectorizer pipeline
search_count = RandomizedSearchCV(
    pipe_count,
    param_distributions,
    n_iter=20,
    cv=cv,
    scoring="f1_macro",
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1,
)

search_count.fit(X_train, y_train)

print(f"Best CountVectorizer F1 (macro): {search_count.best_score_:.4f}")
print(f"Best params: {search_count.best_params_}")

In [ ]:
# Tune TfidfVectorizer pipeline
search_tfidf = RandomizedSearchCV(
    pipe_tfidf,
    param_distributions,
    n_iter=20,
    cv=cv,
    scoring="f1_macro",
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1,
)

search_tfidf.fit(X_train, y_train)

print(f"Best TfidfVectorizer F1 (macro): {search_tfidf.best_score_:.4f}")
print(f"Best params: {search_tfidf.best_params_}")

In [ ]:
# Compare on val
best_count = search_count.best_estimator_
best_tfidf = search_tfidf.best_estimator_

for name, model in [("CountVec (tuned)", best_count), ("TfidfVec (tuned)", best_tfidf)]:
    y_pred = model.predict(X_val)
    acc = accuracy_score(y_val, y_pred)
    f1 = f1_score(y_val, y_pred, average="macro")
    print(f"{name}: Accuracy={acc:.4f}, F1(macro)={f1:.4f}")

## 6. Test Evaluation and Error Analysis


In [ ]:
# Pick best model
val_f1_count = f1_score(y_val, best_count.predict(X_val), average="macro")
val_f1_tfidf = f1_score(y_val, best_tfidf.predict(X_val), average="macro")

if val_f1_tfidf >= val_f1_count:
    best_model = best_tfidf
    best_name = "TfidfVectorizer"
else:
    best_model = best_count
    best_name = "CountVectorizer"

print(f"Best model: {best_name}")

# Test set evaluation
y_pred_test = best_model.predict(X_test)
print(f"Test Accuracy: {accuracy_score(y_test, y_pred_test):.4f}")
print(f"Test F1 (macro): {f1_score(y_test, y_pred_test, average='macro'):.4f}")
print()
print(classification_report(y_test, y_pred_test))

In [ ]:
# Confusion matrix
labels = sorted(y_test.unique())
cm = confusion_matrix(y_test, y_pred_test, labels=labels)

plt.figure(figsize=(12, 10))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=labels, yticklabels=labels,
)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix on Test Set")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Error analysis
errors = pd.DataFrame({
    "text": X_test.values,
    "true_topic": y_test.values,
    "pred_topic": y_pred_test,
})
errors = errors[errors["true_topic"] != errors["pred_topic"]]

print(f"Total misclassified: {len(errors)} ({len(errors)/len(y_test):.1%})")
print()

confusion_pairs = errors.groupby(["true_topic", "pred_topic"]).size().sort_values(ascending=False)
print("Top 10 confusion pairs (true -> predicted):")
print(confusion_pairs.head(10))

In [ ]:
# Examples from top confusion pair
top_pair = confusion_pairs.index[0]
print(f"Examples misclassified as '{top_pair[1]}' instead of '{top_pair[0]}':\n")

pair_errors = errors[(errors["true_topic"] == top_pair[0]) & (errors["pred_topic"] == top_pair[1])]
for _, row in pair_errors.head(3).iterrows():
    print(f"Text: {row['text'][:150]}...")
    print()

In [ ]:
# Save weights
import joblib

weights_dir = os.path.join(REPO_ROOT, "weights", "hw_1")
os.makedirs(weights_dir, exist_ok=True)

weights_path = os.path.join(weights_dir, "best_pipeline.joblib")
joblib.dump(best_model, weights_path)
print(f"Saved best model weights to {weights_path}")


## Conclusions

| Model | Val F1 (macro) | Test F1 (macro) |
|-------|---------------|----------------|
| Dummy | ~ low | ~ low |
| CountVec + LR | see above | - |
| TfidfVec + LR | see above | - |
| CountVec + LR (tuned) | see above | - |
| TfidfVec + LR (tuned) | see above | see above |

Both vectorizers crush the dummy baseline. TF-IDF generally edges out raw counts by downweighting ubiquitous words. Tuning helps moderately -- bigrams and vocab size matter most. Most errors happen between overlapping topics (politics vs economy). Lemmatization is key for Russian.
